In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [9]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Referer": "https://www.google.com/",
    "Upgrade-Insecure-Requests": "1",
}
webpage = requests.get('https://www.linkedin.com/jobs/search/?currentJobId=4440388811&keywords=job&origin=SWITCH_SEARCH_VERTICAL',headers = headers).text

In [19]:
soup = BeautifulSoup(webpage, 'html.parser')

In [20]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta content="d_jobs_guest_search" name="pageKey"/>
  <meta content="max-image-preview:large, noarchive" name="robots"/>
  <meta content="max-image-preview:large, archive" name="bingbot"/>
  <!-- -->
  <meta content="urlType=jserp_custom;emptyResult=false" name="linkedin:pageTag"/>
  <meta content="en_US" name="locale"/>
  <!-- -->
  <meta data-app-version="2.0.2990" data-browser-id="47c0009c-e132-4b52-8390-2f83e58b8495" data-call-tree-id="AAZWjFZHZ3OgND9dIZohcg==" data-dfp-member-lix-treatment="control" data-disable-jsbeacon-pagekey-suffix="false" data-dna-member-lix-treatment="enabled" data-enable-page-view-heartbeat-tracking="" data-human-member-lix-treatment="enabled" data-is-bot="false" data-is-epd-audit-event-enabled="false" data-is-feed-sponsored-tracking-kill-switch-enabled="false" data-member-id="0" data-multiproduct-name="jobs-guest-frontend" data-network-interceptor-lix-value="control" data-page-instance="urn:li:page:d_jobs_guest_s

In [49]:
card = soup.find_all('div', class_='base-card')[3]

# Job title
card.find('h3', class_='base-search-card__title').text.strip()

# Company name
card.find('h4', class_='base-search-card__subtitle').text.strip()

# Location
card.find('span', class_='job-search-card__location').text.strip()

# Posted date (relative)
card.find('time', class_='job-search-card__listdate').text.strip()

# Posted date (actual datetime attribute)
card.find('time', class_='job-search-card__listdate')['datetime']

# Job URL
card.find('a', class_='base-card__full-link')['href'].split('?')[0]

# Job ID
card['data-entity-urn'].split(':')[-1]

# Badge text (Actively Hiring / Be an early applicant)
card.find('span', class_='job-posting-benefits__text').text.strip()

'Be an early applicant'

In [50]:
jobs = []

for card in soup.find_all('div', class_='base-card'):
    title = card.find('h3', class_='base-search-card__title')
    company = card.find('h4', class_='base-search-card__subtitle')
    location = card.find('span', class_='job-search-card__location')
    date_tag = card.find('time', class_='job-search-card__listdate')
    link = card.find('a', class_='base-card__full-link')
    badge = card.find('span', class_='job-posting-benefits__text')

    jobs.append({
        'job_id': card.get('data-entity-urn', '').split(':')[-1],
        'title': title.text.strip() if title else None,
        'company': company.text.strip() if company else None,
        'location': location.text.strip() if location else None,
        'posted_relative': date_tag.text.strip() if date_tag else None,
        'posted_date': date_tag['datetime'] if date_tag else None,
        'badge': badge.text.strip() if badge else None,
        'url': link['href'].split('?')[0] if link else None,
    })

import pandas as pd
df = pd.DataFrame(jobs)
df

,job_id,title,company,location,posted_relative,posted_date,badge,url
0,4059975053,General Apply,LJ Inc.,"Swartz Creek, MI",1 year ago,2024-10-26,Actively Hiring,https://www.linkedin.com/jobs/view/general-app...
1,4359839761,Full-Time and Part-Time Opportunities,Downs Food Group,"Greenville, KY",5 months ago,2026-02-06,Be an early applicant,https://www.linkedin.com/jobs/view/full-time-a...
2,4416827176,Maintenance,Road Ranger,"Amarillo, TX",1 month ago,2026-05-26,Be an early applicant,https://www.linkedin.com/jobs/view/maintenance...
3,4384674895,Cashier,The Home Depot,"Niagara Falls, NY",4 months ago,2026-03-14,Be an early applicant,https://www.linkedin.com/jobs/view/cashier-at-...
4,4384696244,Cashier,The Home Depot,"Kingston, NY",4 months ago,2026-03-14,Be an early applicant,https://www.linkedin.com/jobs/view/cashier-at-...
5,4371016298,General Consideration,CertaSite,"Lafayette, IN",5 months ago,2026-02-10,Be an early applicant,https://www.linkedin.com/jobs/view/general-con...
6,4439025945,Property Meld Join Our Team,Property Meld,"Rapid City, SD",4 weeks ago,2026-06-15,Be an early applicant,https://www.linkedin.com/jobs/view/property-me...
7,4437393280,Maintenance Worker,US Army Corps of Engineers,"South El Monte, CA",4 days ago,2026-07-09,Actively Hiring,https://www.linkedin.com/jobs/view/maintenance...
8,4436156767,Test Role,Atlantic Strategic Minerals,"Petersburg, VA",2 weeks ago,2026-06-30,Be an early applicant,https://www.linkedin.com/jobs/view/test-role-a...
9,4409987737,Sewer I,Lippert,"Elkhart, IN",2 months ago,2026-05-09,Be an early applicant,https://www.linkedin.com/jobs/view/sewer-i-at-...


In [52]:
df.to_csv('linkedin_jobs.csv', index=False)